# BAB IX. Kesimpulan, Temuan, dan Rekomendasi (Communicate Findings and Insights)

Notebook ini menjadi tempat merangkum temuan setelah seluruh tahap sebelumnya dijalankan. Semua angka dan pernyataan faktual harus berasal dari hasil yang dapat direproduksi; jangan mengisi contoh hipotetis sebagai temuan.

## 9.1 Ringkasan temuan utama
Lengkapi setelah analisis:
- Karakteristik dominan dan distribusi variabel.
- Pola atau asosiasi yang teramati.
- Anomali/kandidat pencilan dan keputusan peninjauannya.
- Temuan terkait kualitas data.
- Bukti pendukung berupa tabel/grafik dan nama variabel.

## 9.2 Keterbatasan
Catat keterbatasan sumber, proses ekstraksi, ukuran dan cakupan data, kolom yang tersedia, nilai hilang, ketidakpastian satuan/timestamp, serta keputusan pembersihan dan transformasi.

## 9.3 Langkah berikutnya
Rekomendasi harus sejalan dengan tujuan EDA dan bukti yang tersedia. Jangan merekomendasikan model klasifikasi/regresi tanpa target dan pertanyaan prediktif yang jelas. EDA tidak dengan sendirinya menghasilkan model atau membuktikan penyebab kecelakaan.

## Checklist final laporan
- [ ] Asal-usul dan hak penggunaan data terdokumentasi.
- [ ] Jumlah baris/kolom dan tipe data diverifikasi.
- [ ] Missing values, duplikasi, konsistensi, dan pencilan ditinjau.
- [ ] Statistik dan visualisasi sesuai variabel.
- [ ] Semua transformasi memiliki alasan dan dapat diulang.
- [ ] Temuan didukung tabel/grafik dan tidak melampaui data.
- [ ] Keterbatasan dan keputusan analisis dijelaskan.

In [ ]:
from pathlib import Path
import pandas as pd
from eda_knkt.config import PROCESSED_DATA_DIR
from eda_knkt.ingestion import load_csv
from eda_knkt.cleaning import replace_placeholders
from eda_knkt.transformation import convert_numeric_columns, add_distance_to_stop_feature

DATA_FILE = PROCESSED_DATA_DIR / 'data_logger_knkt_tabel_6_10.csv'
df_raw = load_csv(DATA_FILE)
df_missing = replace_placeholders(df_raw, ['---'])
NUMERIC_OPERATIONAL_COLUMNS = [
    'km', 'm', 'Speed (km/h)', 'Acceleration (m/sec²)',
    'Load (amps)', 'EAB BP (psi)', 'EAB BC (psi)'
]
df_transformed = convert_numeric_columns(
    df_missing, NUMERIC_OPERATIONAL_COLUMNS, errors='coerce'
)
df_transformed = add_distance_to_stop_feature(df_transformed)
df_transformed['Timestamp'] = pd.to_datetime(
    df_transformed['Date'].astype(str) + ' ' + df_transformed['Time'].astype(str),
    format='%d/%m/%Y %H.%M.%S', errors='raise'
)
df_transformed = df_transformed.sort_values('Timestamp').reset_index(drop=True)
df_transformed['Elapsed (s)'] = (
    df_transformed['Timestamp'] - df_transformed['Timestamp'].iloc[0]
).dt.total_seconds()
print(f'Loaded {len(df_raw)} records × {len(df_raw.columns)} columns from {DATA_FILE.name}')

In [ ]:
from pathlib import Path
import pandas as pd
from eda_knkt.config import OUTPUTS_DIR
from eda_knkt.reporting import write_markdown_report

analysis_columns = [
    "distance_to_stop_m", "Speed (km/h)", "Acceleration (m/sec²)",
    "Load (amps)", "EAB BP (psi)", "EAB BC (psi)"
]
summary = df_transformed[analysis_columns].describe().T
summary.insert(0, "n_valid", summary.pop("count").astype(int))
summary_markdown = summary[["n_valid", "mean", "50%", "std", "min", "max"]].round(3).to_string()

category_lines = []
for column in ["Throttle", "PCS Open", "IBS", "DB Excit", "Dir Call", "DB Set", "DB Star"]:
    if column in df_transformed:
        counts = df_transformed[column].value_counts(dropna=True).to_dict()
        category_lines.append(f"- {column}: {counts}")

report_content = f"""# Ringkasan Hasil EDA Data Logger KNKT

## Sumber dan versi data
Transkripsi manual Tabel 6–10 Laporan KNKT KNKT.26.04.03.02.
Dataset analitis: {len(df_transformed)} rekaman unik dan 16 kolom sumber.
Rentang waktu: {df_transformed["Timestamp"].min()} sampai {df_transformed["Timestamp"].max()}.

## Definisi jarak
Kolom sumber km dan m digabung menjadi distance_to_stop_m = km × 1.000 + m.
Besaran ini adalah jarak tersisa menuju titik henti akhir, bukan chainage absolut atau jarak pengereman murni.

## Statistik deskriptif
\n\n{summary_markdown}

## Frekuensi kategori
\n\n{chr(10).join(category_lines)}

## Kualitas dan keterbatasan data
Tujuh placeholder akselerasi tidak dihitung sebagai angka. Empat kolom status memiliki missingness struktural pada Tabel 9–10. Gap waktu 20:51:28–20:51:33 tidak diinterpolasi. Data merupakan satu kejadian dan satu sumber logger; hasil bersifat deskriptif dan tidak membuktikan hubungan kausal.

## Keputusan pembersihan dan transformasi
Data sumber tidak ditimpa. Kolom km dan m dipertahankan; fitur jarak gabungan dihitung pada salinan analisis. Tidak dilakukan imputasi atau winsorization.

## Kesimpulan eksploratif
Ringkasan ini menyajikan statistik dan frekuensi aktual dari pipeline yang dijalankan. Interpretasi operasional harus dibaca bersama laporan KNKT dan batas bukti proyek.
"""

output_path = OUTPUTS_DIR / "eda_findings_summary.md"
write_markdown_report(report_content, output_path)
print("Generated findings summary:", output_path)
